In [ ]:
# find where person.csv actually lives, searching a few levels down
import os

base = os.path.expanduser("~/AI-READI")
for root, dirs, files in os.walk(base):
    for f in files:
        if f == "person.csv":
            print(os.path.join(root, f))

In [ ]:
import pandas as pd
import os

# corrected path - there's a nested clinical_data folder
data_dir = os.path.expanduser("~/AI-READI/clinical_data/clinical_data")
for f in os.listdir(data_dir):
    path = os.path.join(data_dir, f)
    size_mb = os.path.getsize(path) / (1024*1024)
    print(f"{f:40s} {size_mb:8.2f} MB")

In [ ]:
person = pd.read_csv(os.path.join(data_dir, "person.csv"))
print(person.shape)
person.head()

In [ ]:
files = ["visit_occurrence.csv", "condition_occurrence.csv", 
         "procedure_occurrence.csv", "measurement.csv", "observation.csv"]

for fname in files:
    df = pd.read_csv(os.path.join(data_dir, fname), nrows=5)
    print(f"\n=== {fname} ===")
    print(f"columns: {list(df.columns)}")

In [ ]:
# what's actually being measured — the 30 most common measurement types
meas_vals = pd.read_csv(os.path.join(data_dir, "measurement.csv"), 
                          usecols=["measurement_source_value"])
print("=== Top measurement types ===")
print(meas_vals["measurement_source_value"].value_counts().head(30))

# what's actually being observed (surveys, patient-reported data)
obs_vals = pd.read_csv(os.path.join(data_dir, "observation.csv"), 
                         usecols=["observation_source_value"])
print("\n=== Top observation types ===")
print(obs_vals["observation_source_value"].value_counts().head(30))

In [ ]:
pd.set_option('display.max_colwidth', None)

print("=== Diabetes/CV history & socioeconomic observation values ===")
obs_full = obs_vals["observation_source_value"].unique()
for v in sorted(obs_full):
    if any(k in v for k in ["pxfi", "mhoccur", "mhterm", "fh_dm2", "sualc", "susmk", "dmlact", "cl_maristat", "cage,", "brthyy"]):
        print(v)

In [ ]:
target_obs_fields = ["cage", "brthyy", "cl_maristat", 
                      "mhterm_dm1", "mhterm_dm2", "mhterm_predm",
                      "mhoccur_hbp", "mhoccur_clsh", "mhoccur_obs", "mhoccur_mi", 
                      "mhoccur_strk", "mhoccur_circ", "mhoccur_pdr",
                      "fh_dm2pt", "fh_dm2sb",
                      "dmlact", "sualckncf", "susmkncf", "susmkcdur",
                      "pxfi1", "pxfi2", "pxfi3", "pxfi4", "pxfi5"]

# load full observation.csv (need person_id + value + source_value this time)
obs = pd.read_csv(os.path.join(data_dir, "observation.csv"),
                   usecols=["person_id", "observation_source_value", 
                            "value_as_number", "value_as_string", "value_as_concept_id"])

# keep only our target fields, extract the short code before the comma
obs["field_code"] = obs["observation_source_value"].str.split(",").str[0]
obs_filtered = obs[obs["field_code"].isin(target_obs_fields)]

# pivot: one row per participant, one column per field
# prefer value_as_string, fall back to value_as_number
obs_filtered["value"] = obs_filtered["value_as_string"].combine_first(
    obs_filtered["value_as_number"])

registry_obs = obs_filtered.pivot_table(
    index="person_id", columns="field_code", values="value", aggfunc="first")

print(registry_obs.shape)
registry_obs.head()

In [ ]:
obs_filtered = obs[obs["field_code"].isin(target_obs_fields)].copy()

In [ ]:
registry_obs.describe(include='all').T

In [ ]:
# see the actual distinct values (not just top) for suspicious 3-value fields
for col in ["mhoccur_hbp", "mhoccur_circ", "cl_maristat", "pxfi1"]:
    print(col, ":", registry_obs[col].unique())

In [ ]:
# vitals from measurement.csv - same pattern as before
meas = pd.read_csv(os.path.join(data_dir, "measurement.csv"),
                     usecols=["person_id", "measurement_source_value", "value_as_number"])
meas["field_code"] = meas["measurement_source_value"].str.split(",").str[0]

vitals_fields = ["bp1_sysbp_vsorres", "bp1_diabp_vsorres", "pulse_vsorres",
                  "weight_vsorres", "bmi_vsorres", "height_vsorres", 
                  "waist_vsorres", "hip_vsorres", "whr_vsorres"]
meas_filtered = meas[meas["field_code"].isin(vitals_fields)].copy()

registry_vitals = meas_filtered.pivot_table(
    index="person_id", columns="field_code", values="value_as_number", aggfunc="first")
print(registry_vitals.shape)
registry_vitals.head()

In [ ]:
cgm_dir = os.path.expanduser("~/AI-READI/wearable_blood_glucose")

# see the folder structure - don't list everything at once
for root, dirs, files in os.walk(cgm_dir):
    depth = root.replace(cgm_dir, "").count(os.sep)
    if depth <= 2:
        print(root)
        if depth == 2 and files:
            print("   sample files:", files[:3])

In [ ]:
# once you know the real path pattern from above, adjust this:
import glob
sample_files = glob.glob(os.path.join(cgm_dir, "**", "*.tsv"), recursive=True) or \
               glob.glob(os.path.join(cgm_dir, "**", "*.csv"), recursive=True)
print(f"Found {len(sample_files)} CGM files")
print("First file:", sample_files[0])

one_cgm = pd.read_csv(sample_files[0], sep="\t" if sample_files[0].endswith(".tsv") else ",")
print(one_cgm.shape)
one_cgm.head()

In [ ]:
import glob

manifest_path = glob.glob(os.path.join(cgm_dir, "**", "manifest.tsv"), recursive=True)
print(manifest_path)  # sanity check - should print exactly one path

cgm_summary = pd.read_csv(manifest_path[0], sep="\t")
cgm_summary = cgm_summary[["person_id", "average_glucose_level_mg_dl", 
                            "glucose_level_record_count", 
                            "glucose_sensor_sampling_duration_days"]]
cgm_summary = cgm_summary.set_index("person_id")
print(cgm_summary.shape)
cgm_summary.head()

In [ ]:
master = person.set_index("person_id")[["year_of_birth"]]
master = master.join(registry_obs, how="left")
master = master.join(registry_vitals, how="left")
master = master.join(cgm_summary, how="left")

print(master.shape)
master.head()

In [ ]:
sentinel_codes = ["777", "888", "555"]

for col in registry_obs.columns:
    if col not in ["brthyy", "cage"]:
        registry_obs[col] = registry_obs[col].replace(sentinel_codes, pd.NA)

# now redo the merge with the cleaned registry_obs
master = person.set_index("person_id")[["year_of_birth"]]
master = master.join(registry_obs, how="left")
master = master.join(registry_vitals, how="left")
master = master.join(cgm_summary, how="left")

print(master.shape)
master.head()

In [ ]:
# check no sentinel codes remain anywhere
for col in registry_obs.columns:
    bad = master[col].isin(["777", "888", "555"]).sum()
    if bad > 0:
        print(f"{col}: {bad} sentinel values still present!")
print("check complete")

In [ ]:
import os

# save to home directory instead, in a dedicated project folder
output_dir = os.path.expanduser("~/diabetes_registry_project")
os.makedirs(output_dir, exist_ok=True)

master.to_csv(os.path.join(output_dir, "registry_master_v1.csv"))
print("Saved to:", os.path.join(output_dir, "registry_master_v1.csv"))
print("Shape:", master.shape)

In [ ]:
participants = pd.read_csv(os.path.expanduser("~/AI-READI/participants.tsv"), sep="\t")
print(participants.shape)
print(participants.columns.tolist())
participants.head()

In [ ]:
import pandas as pd
import os

master = pd.read_csv(
    os.path.expanduser("~/diabetes_registry_project/registry_master_v1.csv"),
    index_col="person_id"
)
print(master.shape)  # should show (2280, 37)

master = master.drop(columns=["brthyy", "mhterm_dm1"], errors="ignore")
print(master.shape)  # should show (2280, 35)

output_dir = os.path.expanduser("~/diabetes_registry_project")
master.to_csv(os.path.join(output_dir, "registry_master_v2.csv"))
print("Saved:", os.path.join(output_dir, "registry_master_v2.csv"))

In [ ]:
participants = pd.read_csv(os.path.expanduser("~/AI-READI/participants.tsv"), sep="\t")
print(participants.shape)
print(participants.columns.tolist())

In [ ]:
master = master.join(
    participants.set_index("person_id")[["study_group", "clinical_site", "recommended_split"]],
    how="left"
)
print(master.shape)  # should show (2280, 38)
master.head()

In [ ]:
output_dir = os.path.expanduser("~/diabetes_registry_project")
master.to_csv(os.path.join(output_dir, "registry_master_v3copy.csv"))
print("Saved:", master.shape)